# Lab 4 — Il costo misurato dei tuoi automatismi

*Quaderno del capitolo «Perché il tuo cervello non è fatto per questo» di
**Non Fidarti di Me**.*

Il capitolo dice che prendere i piccoli utili e tenere le perdite grandi è
**misurabile e caro**. Qui lo misuri sui tuoi parametri, e poi provi
l'esperimento che il capitolo racconta: distinguere a occhio un processo con un
vantaggio reale da uno senza. Quasi nessuno ci riesce.

In [ ]:
# Setup — esegui questa cella per prima.
%pip install -q "polars>=1.0"
try:
    import avvio
except ModuleNotFoundError:
    import urllib.request

    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/logika-studio/non-fidarti-di-me/main/codice/lab/avvio.py",
        "avvio.py",
    )
    import avvio

avvio.prepara(["btcusdt"])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from cvbook import seed_for
from cvbook.dati import carica
from cvbook.metriche import drawdown_massimo, equity, rendimenti
from cvbook.simulazioni import equity_casuali

prezzi = carica("btcusdt").sort("data")["chiusura"].to_numpy()
r = rendimenti(prezzi)

## 1. Il costo del prendere subito l'utile

Due comportamenti, stessa serie di prezzi, stesso capitale, stessi costi.
Il primo compra e non tocca più niente. Il secondo fa quello che l'esperienza
dei conti reali documenta: chiude appena è in utile di una certa percentuale,
e resta dentro finché la perdita non raggiunge una soglia molto più larga.

Nessuna previsione distingue i due. Solo le due soglie.

In [ ]:
PRENDI_UTILE = 0.10   # ← chiudi quando sei in utile di questa percentuale
SOPPORTA_PERDITA = 0.50  # ← resti dentro finche' la perdita non arriva a questa
COSTO = 0.0012


def con_soglie(p: np.ndarray, su: float, giu: float, costo: float) -> np.ndarray:
    """Chiude quando la posizione tocca una soglia, e rientra il giorno dopo.

    Sono tre le cose che questo comportamento paga rispetto al non far nulla,
    e vale la pena tenerle distinte perche' pesano in modo molto diverso:

    1. il costo dell'uscita e quello del rientro, cioe' due volte `costo`;
    2. il giorno passato fuori dal mercato ad ogni chiusura — ed e' questa la
       voce piu' cara, perche' il capitolo sulla media che mente ha mostrato
       che pochissimi giorni contengono quasi tutto il risultato;
    3. niente altro: nessuna previsione, nessuna scelta di direzione.
    """
    valore = np.empty(len(p))
    valore[0] = 1.0
    ingresso, quota, liquido, dentro = p[0], 1.0 / p[0], 0.0, True

    for i in range(1, len(p)):
        if dentro:
            corrente = quota * p[i]
            variazione = p[i] / ingresso - 1.0
            if variazione >= su or variazione <= -giu:
                liquido = corrente * (1.0 - costo)   # esce: paga il costo
                dentro, corrente = False, liquido
            valore[i] = corrente
        else:
            ingresso = p[i]                          # rientra il giorno dopo
            quota = liquido * (1.0 - costo) / p[i]   # e ripaga il costo
            dentro = True
            valore[i] = quota * p[i]

    return valore


fermo = equity(r)
nervoso = con_soglie(prezzi, PRENDI_UTILE, SOPPORTA_PERDITA, COSTO)

with avvio.figura("schermo"):
    fig, ax = plt.subplots()
    ax.semilogy(fermo, linewidth=1.8, label="compra e non tocca niente")
    ax.semilogy(nervoso, linewidth=1.8, linestyle="--",
                label=f"chiude a +{PRENDI_UTILE:.0%}, sopporta -{SOPPORTA_PERDITA:.0%}")
    ax.set_ylabel("Capitale (scala log)")
    ax.set_xlabel("Giorni")
    ax.legend()
    plt.show()

print(f"chi non ha toccato niente:  {fermo[-1]:6.2f}x   calo massimo {drawdown_massimo(fermo):.1%}")
print(f"chi ha preso i piccoli utili: {nervoso[-1]:6.2f}x   calo massimo {drawdown_massimo(nervoso):.1%}")
print(f"differenza: {nervoso[-1] / fermo[-1] - 1:+.1%}")

Nota le due colonne del calo massimo. Il secondo comportamento ha rinunciato a
una parte del risultato **senza comprarsi in cambio nemmeno un po' di
tranquillità**. Ha pagato per l'illusione di controllo.

## 2. L'asimmetria del dolore, e perché appiattisce

I parametri sperimentali della teoria del prospetto: la perdita pesa circa due
volte e mezzo il guadagno di pari entità, e la curva si appiattisce
allontanandosi dallo zero.

In [ ]:
CURVATURA = 0.88
AVVERSIONE = 2.25

importi = np.linspace(-10_000, 10_000, 400)
# np.where valuta entrambi i rami: si eleva a potenza il valore assoluto e si
# rimette il segno dopo, altrimenti numpy si lamenta delle radici di numeri
# negativi (e ha ragione).
grandezza = np.abs(importi) ** CURVATURA
valore = np.where(importi >= 0, grandezza, -AVVERSIONE * grandezza)

with avvio.figura("schermo"):
    fig, ax = plt.subplots()
    ax.plot(importi, valore, linewidth=2)
    ax.axhline(0, linewidth=0.8, color="#8C8C8C")
    ax.axvline(0, linewidth=0.8, color="#8C8C8C")
    ax.set_xlabel("Guadagno o perdita (euro)")
    ax.set_ylabel("Valore percepito (unità arbitrarie)")
    plt.show()

for x in (2_000, 8_000):
    su = x**CURVATURA
    giu = AVVERSIONE * x**CURVATURA
    print(f"{x:6,d} euro:  piacere {su:9.1f}   dolore {giu:9.1f}   rapporto {giu / su:.2f}")

passo_vicino = AVVERSIONE * (2000**CURVATURA)
passo_lontano = AVVERSIONE * (10000**CURVATURA - 8000**CURVATURA)
print(f"\ndolore nel passare da 0 a -2.000:      {passo_vicino:8.1f}")
print(f"dolore nel passare da -8.000 a -10.000: {passo_lontano:8.1f}")
print("È il motivo per cui, dopo una perdita gia' grande, rischiare ancora "
      "costa pochissimo in termini di sofferenza attesa.")

## 3. Riesci a distinguere il vantaggio dal rumore?

Sei serie. Alcune hanno un vantaggio reale, altre no. Scrivi la tua risposta
prima di eseguire la cella successiva.

In [ ]:
rng = np.random.default_rng(seed_for("lab-bias-indovina"))
VANTAGGI = rng.permutation([0.0, 0.0, 0.0, 0.0005, 0.0005, 0.0])

with avvio.figura("schermo"):
    fig, assi = plt.subplots(2, 3, figsize=(11, 5))
    curve = []
    for k, ax in enumerate(assi.flat):
        c = equity_casuali(1, 400, rendimento_atteso=VANTAGGI[k],
                           volatilita_periodo=0.02, rng=rng)[0]
        curve.append(c)
        ax.plot(c * 100, linewidth=1.4)
        ax.axhline(100, linestyle=":", linewidth=0.8)
        ax.set_title(f"serie {k + 1}", fontsize=10)
        ax.set_xticks([])
    plt.show()

In [ ]:
print("vantaggio reale per operazione:")
for k, v in enumerate(VANTAGGI):
    print(f"  serie {k + 1}: {v:.4%}  →  capitale finale {curve[k][-1]:.2f}x")
print("\nSe le due con vantaggio non sono quelle che avevi indicato, non e' un "
      "tuo limite: 400 osservazioni non bastano a distinguerle, e il capitolo "
      "sulla potenza statistica dice quante ne servirebbero.")

### Esercizi

1. Nella prima cella metti `PRENDI_UTILE = 0.05` e `SOPPORTA_PERDITA = 0.70`:
   è il comportamento estremo, e il costo cresce di conseguenza.
2. Prova `PRENDI_UTILE = 0.50` e `SOPPORTA_PERDITA = 0.10` — cioè il contrario
   di quello che fa quasi tutti. Guarda cosa succede al risultato **e** al calo
   massimo: non è gratis nemmeno quello.
3. Rifai l'esperimento della terza cella cambiando `400` in `4000`. Con dieci
   volte le osservazioni la distinzione diventa possibile. È esattamente il
   punto del capitolo sul potere statistico.